# **0. Carga de datos**

In [26]:
import pandas as pd
import altair as alt
import numpy as np
alt.data_transformers.disable_max_rows()
df = pd.read_csv("data/cleaned_data.csv")

In [10]:
df

,string_id,aq30_id,name_0,name_1,area_km2,bws_score,bwd_score,iav_score,sev_score,drr_score,...,w_awr_fnb_tot_score,w_awr_min_tot_score,w_awr_ong_tot_score,w_awr_smc_tot_score,w_awr_tex_tot_score,continent,Riesgo_Hidrologico_total,Riesgo_Eventos_total,Riesgo_Calidad_Agua_total,Riesgo_por_Industria_total
0,111081-ERI.2_1-3365,89,Eritrea,Debub,6.445810,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.660862,4.421376,4.432915,4.644985,4.625870,Africa,3.890899,2.131406,4.412566,4.499972
1,111081-ERI.6_1-3365,90,Eritrea,Semenawi Keyih Bahri,210.189947,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.660862,4.421376,4.432915,4.644985,4.625870,Africa,3.890899,2.131406,4.412566,4.499972
2,111081-SDN.11_1-1775,92,Sudan,Red Sea,725.925119,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.410905,4.309340,4.314742,4.483448,4.459413,Africa,3.890899,2.131406,4.412566,4.355864
3,111081-SDN.11_1-1930,93,Sudan,Red Sea,345.262750,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.625538,4.374278,4.363119,4.606872,4.561502,Africa,3.890899,2.131406,4.412566,4.465850
4,111081-SDN.11_1-3365,94,Sudan,Red Sea,2989.133981,4.745978,4.708000,3.841307,3.944071,2.215140,...,4.625538,4.374278,4.363119,4.606872,4.561502,Africa,3.890899,2.131406,4.412566,4.465850
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46625,832808-CAN.3_1-352,61194,Canada,Manitoba,1657.645142,0.000000,0.001827,5.000000,2.576914,1.512279,...,0.754779,0.523294,0.309699,0.879260,0.637502,North America,1.818204,0.000000,0.678422,0.672614
46626,832808-CAN.8_1-352,61195,Canada,Nunavut,1468.762076,0.000000,0.001827,5.000000,2.576914,1.512279,...,0.754779,0.523294,0.309699,0.879260,0.637502,North America,1.818204,0.000000,0.678422,0.672614
46627,832809-CAN.12_1-352,61196,Canada,Saskatchewan,174.810225,0.000000,0.000025,3.387390,0.902210,1.473108,...,0.631738,0.412238,0.230250,0.705099,0.527545,North America,1.152547,0.000000,0.676915,0.532449
46628,832809-CAN.3_1-352,61197,Canada,Manitoba,7985.673792,0.000000,0.000025,3.387390,0.902210,1.473108,...,0.631738,0.412238,0.230250,0.705099,0.527545,North America,1.152547,0.000000,0.676915,0.532449


In [20]:
df.columns

Index(['string_id', 'aq30_id', 'name_0', 'name_1', 'area_km2', 'bws_score',
       'bwd_score', 'iav_score', 'sev_score', 'drr_score', 'rfr_score',
       'cfr_score', 'ucw_score', 'cep_score', 'udw_score', 'usa_score',
       'w_awr_agr_tot_score', 'w_awr_che_tot_score', 'w_awr_con_tot_score',
       'w_awr_elp_tot_score', 'w_awr_fnb_tot_score', 'w_awr_min_tot_score',
       'w_awr_ong_tot_score', 'w_awr_smc_tot_score', 'w_awr_tex_tot_score',
       'continent', 'Riesgo_Hidrologico_total', 'Riesgo_Eventos_total',
       'Riesgo_Calidad_Agua_total', 'Riesgo_por_Industria_total'],
      dtype='object')

# **1. Ecuador en el contexto global del estrés hídrico**

## **Contexto de la pregunta analítica**

Para comprender la magnitud del reto hídrico en Ecuador, analizamos su posición dentro de la distribución mundial. El siguiente gráfico muestra el **Estrés Hídrico Base (Baseline Water Stress)** ponderado por área de todos los países, agrupados por continente. Esta vista panorámica permite contrastar la realidad ecuatoriana no solo a nivel global, sino frente al comportamiento de Sudamérica.

## **Visualización**

In [36]:
# Configuración global de estilo para Altair (Altair 5.5.0+)
alt.theme.enable('fivethirtyeight')

# 2. Función de promedio ponderado
def w_avg(df, values_cols, weight_col):
    d = df[values_cols]
    w = df[weight_col]
    return (d.multiply(w, axis=0).sum(axis=0)) / w.sum()

# 3. Agrupación provincial
cols_riesgo = [col for col in df.columns if col.endswith('_score') or col.endswith('_total')]
df_prov = df.groupby('name_1').apply(lambda x: w_avg(x, cols_riesgo, 'area_km2')).reset_index()

# 4. Dimensiones macro
cols_hidro = ['bws_score', 'bwd_score', 'iav_score', 'sev_score', 'drr_score']

cols_eventos = ['rfr_score', 'cfr_score']

cols_calidad = ['ucw_score', 'cep_score', 'udw_score', 'usa_score']

cols_industria = [col for col in df.columns if col.startswith('w_awr_')]

df_prov['Riesgo_Total'] = df[['Riesgo_Hidrologico_total', 'Riesgo_Eventos_total','Riesgo_Calidad_Agua_total', 'Riesgo_por_Industria_total']].mean(axis=1)

C:\Users\zevBetaUltra7\AppData\Local\Temp\ipykernel_138904\3698906552.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_prov = df.groupby('name_1').apply(lambda x: w_avg(x, cols_riesgo, 'area_km2')).reset_index()


In [37]:
import pandas as pd
import altair as alt
import numpy as np

# Configuración global
alt.theme.enable('fivethirtyeight')

# 1. Cargar el dataset global
df_global = pd.read_csv("data/cleaned_data_long_format.csv") # Cambia por tu archivo real

# 2. Filtrar únicamente el indicador de Estrés Hídrico Base
df_bws = df_global[df_global['score'] == 'bws_score'].copy()

# 3. Función de promedio ponderado global
def w_avg_global(df, val_col, weight_col):
    if df[weight_col].sum() == 0: 
        return 0
    return (df[val_col] * df[weight_col]).sum() / df[weight_col].sum()

# 4. Agrupar la data para calcular el score final de cada PAÍS del mundo
df_paises = df_bws.groupby(['name_0', 'continent']).apply(
    lambda x: w_avg_global(x, 'value', 'area_km2')
).reset_index(name='Estres_Hidrico')

df_paises['continent'] = df_paises['continent'].replace({'South America': 'Sudamérica', 'North America': 'Norteamérica'})

# Crear 'Jitter' (ruido vertical) para evitar solapamiento
np.random.seed(42)
df_paises['jitter'] = np.random.uniform(-0.0, 0.0, len(df_paises))

# 5. Construcción de la Visualización
base = alt.Chart(df_paises).encode(
    x=alt.X('Estres_Hidrico:Q', title='Estrés Hídrico Base (0-5)', scale=alt.Scale(domain=[-0.2, 5.2])),
    y=alt.Y('continent:N', title='Continente', sort='ascending', axis=alt.Axis(grid=True, labelFontSize=12))
)

# Capa 1: Puntos
puntos = base.mark_circle(stroke='white', strokeWidth=0.5).encode(
    yOffset='jitter:Q', 
    
    # ---------------------------------------------------------
    # LA MAGIA ARQUITECTÓNICA: Condición con Escala
    # Si es Ecuador -> Rojo Intenso. Si NO -> Color por Continente (paleta set2)
    # ---------------------------------------------------------
    color=alt.condition(
        alt.datum.name_0 == 'Ecuador',
        alt.value('#cc0000'),  # Valor fijo para Ecuador (Rojo Intenso)
        alt.Color('continent:N', scale=alt.Scale(scheme='set2'), legend=None) # Escala de colores para el resto
    ),
    
    size=alt.condition(
        alt.datum.name_0 == 'Ecuador',
        alt.value(300),          # Hacemos a Ecuador aún más grande para que domine
        alt.value(70)
    ),
    opacity=alt.condition(
        alt.datum.name_0 == 'Ecuador',
        alt.value(1.0),          # Ecuador sólido
        alt.value(0.7)           # Resto de países ligeramente transparentes
    ),
    tooltip=[
        alt.Tooltip('name_0:N', title='País'),
        alt.Tooltip('continent:N', title='Continente'),
        alt.Tooltip('Estres_Hidrico:Q', title='Estrés Hídrico', format='.2f')
    ]
)

# Capa 2: Etiqueta de Ecuador
texto_ecuador = base.mark_text(
    align='left', 
    baseline='middle', 
    dx=14,           
    fontSize=13, 
    fontWeight='bold', 
    color='#cc0000' # Mismo color que el punto
).transform_filter(
    alt.datum.name_0 == 'Ecuador'
).encode(
    yOffset='jitter:Q', 
    text='name_0:N'
)

# Capa 3: Línea y etiqueta de promedio mundial
promedio_mundial = df_paises['Estres_Hidrico'].mean()

linea_promedio = alt.Chart(pd.DataFrame({'x': [promedio_mundial]})).mark_rule(
    strokeDash=[5, 5], color='#4A4A4A', strokeWidth=1.5
).encode(x='x:Q')

texto_promedio = alt.Chart(pd.DataFrame({'x': [promedio_mundial]})).mark_text(
    align='left', baseline='bottom', dx=5, dy=-5, 
    text='Promedio Mundial', color='#4A4A4A', fontSize=11, fontWeight='bold'
).encode(
    x='x:Q', 
    y=alt.value(0) 
)

# Ensamblaje final
grafico_contexto = (puntos + texto_ecuador + linea_promedio + texto_promedio).properties(
    title=alt.TitleParams(
        text='Posicionamiento Global del Estrés Hídrico',
        subtitle='Cada punto representa un país.',
        color='#333333',
        dy=-10
    ),
    width=650, 
    height=350
).interactive()

grafico_contexto

C:\Users\zevBetaUltra7\AppData\Local\Temp\ipykernel_138904\2943373763.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_paises = df_bws.groupby(['name_0', 'continent']).apply(


alt.LayerChart(...)

## **Interpretación**

# **2. Riesgo hidrológico por provincia**

## **Contexto de la pregunta analítica**

## **Visualización**

In [34]:
# mapear el pais 

,name_1,bws_score,bwd_score,iav_score,sev_score,drr_score,rfr_score,cfr_score,ucw_score,cep_score,...,w_awr_fnb_tot_score,w_awr_min_tot_score,w_awr_ong_tot_score,w_awr_smc_tot_score,w_awr_tex_tot_score,Riesgo_Hidrologico_total,Riesgo_Eventos_total,Riesgo_Calidad_Agua_total,Riesgo_por_Industria_total,Riesgo_Total
0,!Karas,4.962610,4.945965,4.073791,1.815123,1.332834,1.940332,0.000000,3.948875,2.063849,...,4.301177,3.486966,3.447741,4.150757,3.974128,3.426065,0.970166,3.358051,3.921115,3.733711
1,Aargau,0.164204,0.237953,1.104301,0.340277,2.202001,0.696723,0.000000,0.132120,3.373784,...,0.849949,0.624881,0.351677,0.778546,0.596241,0.809747,0.348362,0.876476,0.693064,3.733711
2,Abia,0.000000,0.033012,0.828860,1.269267,3.402744,3.476094,3.178393,5.000000,1.189432,...,2.857677,4.235960,4.387745,4.009644,4.204618,1.106777,3.327243,3.925666,3.622128,3.697684
3,Abidjan,0.000000,0.065531,1.273050,0.906364,2.569847,2.335892,0.000000,5.000000,1.363962,...,2.498721,3.616042,4.013291,3.638373,4.034645,0.962958,1.167946,3.698674,2.968636,3.725180
4,Abkhazia,0.045183,0.253938,1.070064,1.022039,2.086577,3.961601,0.000000,5.000000,2.113987,...,1.888693,2.569743,2.263297,3.120019,3.018480,0.895560,1.980801,3.130366,2.303395,3.725180
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2846,Štip,2.995955,1.868017,1.777869,1.157241,3.182766,2.656972,0.092774,5.000000,1.804892,...,3.546871,2.501459,1.719871,3.789027,3.417170,2.196370,1.374873,2.318655,3.001007,3.183672
2847,Šumadijski,2.399835,1.946068,1.541290,0.814807,3.633366,2.542815,0.000000,3.019152,3.679651,...,3.022881,2.264575,1.385892,3.119264,2.587612,2.067073,1.271408,2.265311,2.585934,2.909007
2848,Šuto Orizari,2.995955,1.868017,1.777869,1.157241,3.182766,2.656972,0.092774,5.000000,1.804892,...,3.546871,2.501459,1.719871,3.789027,3.417170,2.196370,1.374873,2.318655,3.001007,2.909007
2849,Želino,2.995955,1.868017,1.777869,1.157241,3.182766,2.656972,0.092774,5.000000,1.804892,...,3.546871,2.501459,1.719871,3.789027,3.417170,2.196370,1.374873,2.318655,3.001007,2.775267


In [42]:
import pandas as pd
import altair as alt
import numpy as np

# Configuración global de Altair
alt.data_transformers.disable_max_rows()
alt.theme.enable('fivethirtyeight')

# 1. Cargar datos
df = pd.read_csv("data/cleaned_data.csv")

# --- CORRECCIÓN 1: Filtrar únicamente Ecuador antes de agrupar ---
df_ecuador = df[df['name_0'] == 'Ecuador'].copy()

# 2. Función de promedio ponderado
def w_avg(df, values_cols, weight_col):
    d = df[values_cols]
    w = df[weight_col]
    return (d.multiply(w, axis=0).sum(axis=0)) / w.sum()

# 3. Agrupación provincial
cols_riesgo = [col for col in df_ecuador.columns if col.endswith('_score') or col.endswith('_total')]
df_prov = df_ecuador.groupby('name_1').apply(lambda x: w_avg(x, cols_riesgo, 'area_km2')).reset_index()

# 4. Dimensiones macro
cols_hidro = ['bws_score', 'bwd_score', 'iav_score', 'sev_score', 'drr_score']
cols_eventos = ['rfr_score', 'cfr_score']
cols_calidad = ['ucw_score', 'cep_score', 'udw_score', 'usa_score']
cols_industria = [col for col in df_ecuador.columns if col.startswith('w_awr_')]

# --- CORRECCIÓN 2: Usar 'df_prov' en lugar de 'df' para calcular el promedio final ---
df_prov['Riesgo_Total'] = df_prov[['Riesgo_Hidrologico_total', 'Riesgo_Eventos_total', 'Riesgo_Calidad_Agua_total', 'Riesgo_por_Industria_total']].mean(axis=1)

# =========================================================================
# --- CELDA ACTUALIZADA: Gráfico Interactivo con Filtro Base Cero ---
# =========================================================================

# --- CORRECCIÓN 3: Ajustar el nombre correcto de la columna en 'id_vars' ---
df_hidro_melt = df_prov.melt(
    id_vars=['name_1', 'Riesgo_Hidrologico_total'], 
    value_vars=cols_hidro, 
    var_name='Indicador', 
    value_name='Score'
)

# Renombrar indicadores para legibilidad
dict_hidro = {
    'bws_score': 'Estrés Base', 
    'bwd_score': 'Agotamiento', 
    'iav_score': 'Var. Interanual', 
    'sev_score': 'Var. Estacional', 
    'drr_score': 'Riesgo Sequía'
}
df_hidro_melt['Indicador'] = df_hidro_melt['Indicador'].map(dict_hidro)

# Lista fija de los indicadores (Truco para que la leyenda no desaparezca al filtrar)
lista_indicadores = list(dict_hidro.values())

# 1. Parámetro de interacción (bind='legend' mantiene la selección desde la leyenda)
seleccion_indicador = alt.selection_point(fields=['Indicador'], bind='legend')

# 2. Construir el gráfico
bars_hidro = alt.Chart(df_hidro_melt).mark_bar(
    cornerRadiusEnd=3, 
    stroke='white',    
    strokeWidth=0.5
).encode(
    x=alt.X('sum(Score):Q', title='Score Acumulado (0.0 a Base)'),
    
    # El sort se recalculará dinámicamente según el filtro aplicado
    y=alt.Y('name_1:N', sort=alt.EncodingSortField(field='Score', op='sum', order='descending'), title='Provincia'),
    
    # IMPORTANTE: Definimos el 'domain' explícitamente dentro de Scale
    color=alt.Color('Indicador:N', 
                    scale=alt.Scale(scheme='reds', domain=lista_indicadores),
                    legend=alt.Legend(title="Indicador (Clic para filtrar)", symbolType='circle')),
    
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Indicador:N', title='Sub-indicador'),
        alt.Tooltip('Score:Q', title='Score', format='.2f')
    ]
).properties(
    title=alt.TitleParams(
        text='Composición del Riesgo Hidrológico por Provincia',
        subtitle='Interactividad: Haz clic en la leyenda para filtrar. Las barras se apilarán desde cero.',
        color='#333333'
    ),
    width=650, 
    height=500
).add_params(
    seleccion_indicador
).transform_filter(
    seleccion_indicador
).interactive()

# Mostrar gráfico
bars_hidro

C:\Users\zevBetaUltra7\AppData\Local\Temp\ipykernel_138904\4254363629.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_prov = df_ecuador.groupby('name_1').apply(lambda x: w_avg(x, cols_riesgo, 'area_km2')).reset_index()


alt.Chart(...)

## **Interpretación**

# **3. Eventos hidrológicos**

## **Contexto de la pregunta analítica**

Implementamos un **Dumbbell Plot (Gráfico de Pesas)**. Esta visualización avanzada es perfecta para mostrar la brecha o diferencia exacta entre dos variables dentro de una misma categoría.

## **Visualización**

In [46]:
# =========================================================================
# --- CELDA ACTUALIZADA: Dumbbell Plot (Fluvial vs Costera) ---
# =========================================================================

# 1. Melt: Añadimos 'Riesgo_Eventos_total' a id_vars para usarlo como criterio de orden
df_eventos_melt = df_prov.melt(
    id_vars=['name_1', 'Riesgo_Eventos_total'], 
    value_vars=['rfr_score', 'cfr_score'], 
    var_name='Tipo_Inundacion', 
    value_name='Score'
)

# 2. Renombrar para legibilidad en la visualización
df_eventos_melt['Tipo_Inundacion'] = df_eventos_melt['Tipo_Inundacion'].replace({
    'rfr_score': 'Fluvial (Ríos)', 
    'cfr_score': 'Costera'
})

# 3. Construcción del Dumbbell Plot
# CAPA 1: La línea que conecta los puntos (la "barra" de la pesa)
rule = alt.Chart(df_eventos_melt).mark_rule(color='#c0c0c0', strokeWidth=2.5).encode(
    x=alt.X('min(Score):Q', title='Score de Riesgo (0-5)', scale=alt.Scale(domain=[0, 5])),
    x2=alt.X2('max(Score):Q'),
    
    # Ordenamos el Eje Y basándonos en el Riesgo Total de Eventos (descendente)
    y=alt.Y('name_1:N', 
            sort=alt.EncodingSortField(field='Riesgo_Eventos_total', op='max', order='descending'), 
            title='Provincia', 
            axis=alt.Axis(grid=True)) # Añadimos grid horizontal para guiar el ojo
)

# CAPA 2: Los puntos (las "pesas")
points = alt.Chart(df_eventos_melt).mark_circle(size=180, opacity=1).encode(
    x=alt.X('Score:Q'),
    y=alt.Y('name_1:N', sort=alt.EncodingSortField(field='name_1', op='max', order='descending')),
    
    color=alt.Color('Tipo_Inundacion:N', 
                    scale=alt.Scale(domain=['Fluvial (Ríos)', 'Costera'], range=['#1f77b4', '#ff7f0e']),
                    legend=alt.Legend(title="Tipo de Inundación", orient='top-right', fillColor='white', padding=10)),
    
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Tipo_Inundacion:N', title='Tipo de Inundación'),
        alt.Tooltip('Score:Q', title='Score Específico', format='.2f'),
        # Al incluir el total en el tooltip, damos más contexto al usuario
        alt.Tooltip('Riesgo_Eventos_total:Q', title='Riesgo Total de Eventos', format='.2f') 
    ]
)

# Ensamblaje final
dumbbell = (rule + points).properties(
    title=alt.TitleParams(
        text='Brecha de Riesgo: Inundaciones Fluviales vs. Costeras',
        subtitle='Provincias ordenadas por su exposición total a eventos extremos.',
        color='#333333'
    ),
    width=650, 
    height=550
).interactive()

# Mostrar gráfico
dumbbell

alt.LayerChart(...)

## **Interpretación**

# **4. Calidad del agua**

## **Contexto de la pregunta analítica**

## **Visualización**

In [55]:
# =========================================================================
# --- CELDA ACTUALIZADA: Matriz de Vulnerabilidad (Calidad vs Cantidad) ---
# =========================================================================

# 1. Calculamos las medianas para crear los 4 cuadrantes analíticos
med_hidro = df_prov['Riesgo_Hidrologico_total'].median()
med_calidad = df_prov['Riesgo_Calidad_Agua_total'].median()

# 2. Construcción de la Matriz de Dispersión (Scatter Plot)
scatter = alt.Chart(df_prov).mark_circle(
    opacity=0.85, 
    stroke='white', 
    strokeWidth=1.5
).encode(
    # Eje X: Riesgo Hidrológico (Fijamos escala 0-5)
    x=alt.X('Riesgo_Hidrologico_total:Q', 
            title='Riesgo Hidrológico Total (Escasez / Estrés)', 
            scale=alt.Scale(domain=[0, 5])),
            
    # Eje Y: Calidad del Agua (Fijamos escala 0-5)
    y=alt.Y('Riesgo_Calidad_Agua_total:Q', 
            title='Riesgo en Calidad de Agua (Contaminación / Falta de Acceso)', 
            scale=alt.Scale(domain=[0, 5])),
            
    # Tamaño de la burbuja: Refleja el Riesgo Total de la provincia para dar peso visual
    size=alt.Size('Riesgo_Total:Q', 
                  scale=alt.Scale(range=[100, 800]), 
                  legend=None),
                  
    # Color: Mapeamos la severidad de la calidad del agua (Gradiente)
    color=alt.Color('Riesgo_Calidad_Agua_total:Q', 
                    scale=alt.Scale(scheme='viridis'), 
                    legend=alt.Legend(title="Severidad (Calidad)", orient='top-right')),
                    
    # Tooltip: Aquí resolvemos el desglose de los servicios básicos
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Riesgo_Calidad_Agua_total:Q', title='Riesgo Calidad (Total)', format='.2f'),
        alt.Tooltip('Riesgo_Hidrologico_total:Q', title='Riesgo Hidrológico', format='.2f'),
        alt.Tooltip('usa_score:Q', title='Riesgo: Falta de Saneamiento', format='.2f'),
        alt.Tooltip('udw_score:Q', title='Riesgo: Falta Agua Potable', format='.2f'),
        alt.Tooltip('ucw_score:Q', title='Riesgo: Aguas Residuales', format='.2f')
    ]
)

# 3. Líneas divisorias (Cuadrantes) basadas en la mediana nacional
vline = alt.Chart(pd.DataFrame({'x': [med_hidro]})).mark_rule(strokeDash=[5, 5], color='#888888', strokeWidth=1.5).encode(x='x:Q')
hline = alt.Chart(pd.DataFrame({'y': [med_calidad]})).mark_rule(strokeDash=[5, 5], color='#888888', strokeWidth=1.5).encode(y='y:Q')

# 4. Anotaciones de los cuadrantes (Opcional pero altamente recomendado para público general)
cuadrantes_df = pd.DataFrame({
    'x': [4.8, 0.2, 0.2, 4.8],
    'y': [4.8, 4.8, 0.2, 0.2],
    'texto': ['Crítico (Ambos)', 'Mala Calidad / Con Agua', 'Bajo Riesgo', 'Escasez / Buena Calidad'],
    'lado': ['right', 'left', 'left', 'right']
})

textos_left = alt.Chart(cuadrantes_df[cuadrantes_df['lado'] == 'left']).mark_text(
    fontSize=10, color='gray', opacity=0.7, align='left'
).encode(
    x='x:Q',
    y='y:Q',
    text='texto:N'
)

textos_right = alt.Chart(cuadrantes_df[cuadrantes_df['lado'] == 'right']).mark_text(
    fontSize=10, color='gray', opacity=0.7, align='right'
).encode(
    x='x:Q',
    y='y:Q',
    text='texto:N'
)

textos_cuadrantes = textos_left + textos_right

# 5. Ensamblaje Final
matriz_calidad = (scatter + vline + hline + textos_cuadrantes).properties(
    title=alt.TitleParams(
        text='Matriz de Vulnerabilidad: Calidad vs. Cantidad de Agua',
        subtitle='Relación entre la presión hidrológica estructural y el acceso a servicios básicos.',
        color='#333333'
    ),
    width=600,
    height=550
).interactive()

# Mostrar gráfico
matriz_calidad

alt.LayerChart(...)

## **Interpretación**

# **5. Riesgo por industria**

## **Contexto de la pregunta analítica**

Para identificar qué provincias concentran altos niveles de riesgo en múltiples dimensiones y sectores, empleamos **Mapas de Calor (Heatmaps)**. Esto nos permite una lectura matricial rápida de los territorios más críticos.

## **Visualización**

In [48]:
# =========================================================================
# --- CELDA ACTUALIZADA: Heatmap de Riesgo por Sector Industrial ---
# =========================================================================

# 1. Melt: Añadimos 'Riesgo_por_Industria_total' a id_vars para usarlo como criterio de orden
df_ind_melt = df_prov.melt(
    id_vars=['name_1', 'Riesgo_por_Industria_total'], 
    value_vars=cols_industria, 
    var_name='Sector', 
    value_name='Score'
)

# 2. Diccionario y mapeo para legibilidad
dict_sectores = {
    'w_awr_agr_tot_score': 'Agricultura', 
    'w_awr_che_tot_score': 'Química', 
    'w_awr_con_tot_score': 'Construcción',
    'w_awr_elp_tot_score': 'Energía', 
    'w_awr_fnb_tot_score': 'Alimentos', 
    'w_awr_min_tot_score': 'Minería',
    'w_awr_ong_tot_score': 'Petróleo/Gas', 
    'w_awr_smc_tot_score': 'Semiconductores', 
    'w_awr_tex_tot_score': 'Textil'
}
df_ind_melt['Sector'] = df_ind_melt['Sector'].map(dict_sectores)

# 3. Construcción del Heatmap
heat_ind = alt.Chart(df_ind_melt).mark_rect(
    stroke='white',      # Añade una fina línea blanca entre celdas (Best Practice)
    strokeWidth=0.5
).encode(
    x=alt.X('Sector:N', 
            title='Sector Económico', 
            axis=alt.Axis(labelAngle=-45, labelFontSize=11)),
            
    # Ordenamos el Eje Y basándonos en el Riesgo Industrial Total (descendente)
    y=alt.Y('name_1:N', 
            sort=alt.EncodingSortField(field='Riesgo_por_Industria_total', op='max', order='descending'), 
            title='Provincia',
            axis=alt.Axis(labelFontSize=11)),
    
    # Mantenemos tu excelente decisión de anclar el domain=[0, 5]
    color=alt.Color('Score:Q', 
                    scale=alt.Scale(scheme='inferno', reverse=True, domain=[0, 5]), 
                    title='Riesgo (0-5)',
                    legend=alt.Legend(orient='right', padding=10)),
                    
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Sector:N', title='Sector Económico'),
        alt.Tooltip('Score:Q', title='Score Sectorial', format='.2f'),
        # Añadimos el promedio global industrial al tooltip para dar más contexto
        alt.Tooltip('Riesgo_por_Industria_total:Q', title='Riesgo Industrial Promedio', format='.2f')
    ]
).properties(
    title=alt.TitleParams(
        text='Riesgo Hídrico por Sector Industrial',
        subtitle='Provincias ordenadas por su exposición promedio total del sector industria.',
        color='#333333'
    ),
    width=550,  # Un poco más ancho para que los nombres inclinados respiren bien
    height=550  # Un poco más alto para que las celdas sean más cuadradas
).interactive()

# Mostrar el gráfico
heat_ind

alt.Chart(...)

## **Interpretación**

# **6. Integración del análisis**

## **Contexto de la pregunta analítica**

## **Visualización**

In [52]:
df.columns

Index(['string_id', 'aq30_id', 'name_0', 'name_1', 'area_km2', 'bws_score',
       'bwd_score', 'iav_score', 'sev_score', 'drr_score', 'rfr_score',
       'cfr_score', 'ucw_score', 'cep_score', 'udw_score', 'usa_score',
       'w_awr_agr_tot_score', 'w_awr_che_tot_score', 'w_awr_con_tot_score',
       'w_awr_elp_tot_score', 'w_awr_fnb_tot_score', 'w_awr_min_tot_score',
       'w_awr_ong_tot_score', 'w_awr_smc_tot_score', 'w_awr_tex_tot_score',
       'continent', 'Riesgo_Hidrologico_total', 'Riesgo_Eventos_total',
       'Riesgo_Calidad_Agua_total', 'Riesgo_por_Industria_total'],
      dtype='object')

In [53]:
bubble_integrated = alt.Chart(df_prov).mark_circle(stroke='white', strokeWidth=1, opacity=0.8).encode(
    x=alt.X('Riesgo_Hidrologico_total:Q', title='Riesgo Hidrológico', scale=alt.Scale(zero=False)),
    y=alt.Y('Riesgo_Calidad_Agua_total:Q', title='Riesgo en Calidad de Agua', scale=alt.Scale(zero=False)),
    size=alt.Size('Riesgo_Eventos_total:Q', title='Eventos Extremos', scale=alt.Scale(range=[50, 800])),
    color=alt.Color('Riesgo_por_Industria_total:Q', title='Riesgo Industrial', scale=alt.Scale(scheme='viridis')),
    tooltip=[
        alt.Tooltip('name_1:N', title='Provincia'),
        alt.Tooltip('Riesgo_Total:Q', title='Riesgo Global Promedio', format='.2f'),
        alt.Tooltip('Riesgo_Hidrologico_total:Q', format='.2f'),
        alt.Tooltip('Riesgo_Calidad_Agua_total:Q', format='.2f'),
        alt.Tooltip('Riesgo_Eventos_total:Q', format='.2f'),
        alt.Tooltip('Riesgo_por_Industria_total:Q', format='.2f')
    ]
).properties(
    title='Visión Integrada: Las 4 Dimensiones del Riesgo Hídrico',
    width=650, height=500
)

# Añadimos cuadrantes basados en la mediana para perfilar
vline = alt.Chart(pd.DataFrame({'x': [df_prov['Riesgo_Hidrologico_total'].median()]})).mark_rule(strokeDash=[3,3]).encode(x='x:Q')
hline = alt.Chart(pd.DataFrame({'y': [df_prov['Riesgo_Calidad_Agua_total'].median()]})).mark_rule(strokeDash=[3,3]).encode(y='y:Q')

(bubble_integrated + vline + hline).interactive()

alt.LayerChart(...)

## **Interpretación**